# Knowledge Graph Results Viewer

Read-only notebook for inspecting knowledge-graph outputs after a pipeline run.

It loads tables from a project directory like `outputs/kg_cia_smoke_para50/` and previews:

- `extraction/`
- `graphs/`
- `hierarchy/`
- optional inline HTML visualizations from `hierarchy/visualizations/`


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import HTML, Markdown, display

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_columns", 20)


def find_repo_root(start=None):
    current = (Path.cwd() if start is None else Path(start)).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "knowledge_graph" / "run_pipeline.py").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing knowledge_graph/run_pipeline.py")


def load_csv(path):
    if not path.exists():
        print(f"Missing: {path}")
        return pd.DataFrame()
    df = pd.read_csv(path)
    print(f"Loaded {path} -> {df.shape}")
    return df


def preview(df, preferred_cols=None, n=10):
    if df.empty:
        print("Empty table")
        return
    preferred_cols = preferred_cols or []
    cols = [col for col in preferred_cols if col in df.columns]
    display(df[cols].head(n) if cols else df.head(n))


def rel(path, repo_root):
    try:
        return path.resolve().relative_to(repo_root.resolve()).as_posix()
    except Exception:
        return str(path)


In [2]:
REPO_ROOT = find_repo_root()

# Change this to the project directory you want to inspect.
PROJECT_DIR = REPO_ROOT / "outputs" / "kg_cia_full_para_v2"

# Set True if you want to inline the generated PyVis HTML inside the notebook.
SHOW_HTML_VISUALIZATIONS = False

print("REPO_ROOT:", REPO_ROOT)
print("PROJECT_DIR:", PROJECT_DIR)

if not PROJECT_DIR.exists():
    raise FileNotFoundError(f"Project directory does not exist: {PROJECT_DIR}")


REPO_ROOT: /Users/jimyhc/Desktop/research/IARPA
PROJECT_DIR: /Users/jimyhc/Desktop/research/IARPA/outputs/kg_cia_full_para_v2


In [3]:
table_paths = {
    "entities": PROJECT_DIR / "extraction" / "entities.csv",
    "events": PROJECT_DIR / "extraction" / "events.csv",
    "claims": PROJECT_DIR / "extraction" / "claims.csv",
    "relations": PROJECT_DIR / "extraction" / "relations.csv",
    "canonical_nodes": PROJECT_DIR / "graphs" / "canonical_nodes_cluster.csv",
    "analysis_edges": PROJECT_DIR / "graphs" / "analysis_edges_cluster.csv",
    "node_to_level1": PROJECT_DIR / "graphs" / "node_to_level1.csv",
    "level1_summary": PROJECT_DIR / "graphs" / "level1_summary.csv",
    "node_to_level2": PROJECT_DIR / "graphs" / "node_to_level2.csv",
    "level2_summary": PROJECT_DIR / "graphs" / "level2_summary.csv",
    "community_nodes": PROJECT_DIR / "hierarchy" / "community_nodes.csv",
    "community_edges": PROJECT_DIR / "hierarchy" / "community_edges.csv",
    "community_to_narrative": PROJECT_DIR / "hierarchy" / "community_to_narrative.csv",
    "narrative_nodes": PROJECT_DIR / "hierarchy" / "narrative_nodes.csv",
    "narrative_edges": PROJECT_DIR / "hierarchy" / "narrative_edges.csv",
    "hierarchy_index": PROJECT_DIR / "hierarchy" / "hierarchy_index.csv",
}

tables = {name: load_csv(path) for name, path in table_paths.items()}


Loaded /Users/jimyhc/Desktop/research/IARPA/outputs/kg_cia_full_para_v2/extraction/entities.csv -> (90614, 12)
Loaded /Users/jimyhc/Desktop/research/IARPA/outputs/kg_cia_full_para_v2/extraction/events.csv -> (26743, 13)
Loaded /Users/jimyhc/Desktop/research/IARPA/outputs/kg_cia_full_para_v2/extraction/claims.csv -> (25995, 13)
Loaded /Users/jimyhc/Desktop/research/IARPA/outputs/kg_cia_full_para_v2/extraction/relations.csv -> (35036, 11)
Loaded /Users/jimyhc/Desktop/research/IARPA/outputs/kg_cia_full_para_v2/graphs/canonical_nodes_cluster.csv -> (41870, 14)
Loaded /Users/jimyhc/Desktop/research/IARPA/outputs/kg_cia_full_para_v2/graphs/analysis_edges_cluster.csv -> (339485, 9)
Loaded /Users/jimyhc/Desktop/research/IARPA/outputs/kg_cia_full_para_v2/graphs/node_to_level1.csv -> (41870, 2)
Loaded /Users/jimyhc/Desktop/research/IARPA/outputs/kg_cia_full_para_v2/graphs/level1_summary.csv -> (62, 7)
Loaded /Users/jimyhc/Desktop/research/IARPA/outputs/kg_cia_full_para_v2/graphs/node_to_level2.c

In [4]:
summary_rows = []
for name, df in tables.items():
    summary_rows.append({"table": name, "rows": len(df), "columns": len(df.columns)})

summary_df = pd.DataFrame(summary_rows).sort_values(["rows", "table"], ascending=[False, True]).reset_index(drop=True)
display(summary_df)

high_level = {
    "unique_entity_labels": tables["entities"]["label"].nunique() if "label" in tables["entities"].columns else 0,
    "unique_event_labels": tables["events"]["label"].nunique() if "label" in tables["events"].columns else 0,
    "unique_docs_in_entities": tables["entities"]["doc_id"].nunique() if "doc_id" in tables["entities"].columns else 0,
    "communities": len(tables["community_nodes"]),
    "narratives": len(tables["narrative_nodes"]),
}
display(pd.DataFrame([high_level]))


,table,rows,columns
0,analysis_edges,339485,9
1,entities,90614,12
2,canonical_nodes,41870,14
3,node_to_level1,41870,2
4,relations,35036,11
5,events,26743,13
6,claims,25995,13
7,community_edges,3393,8
8,community_nodes,2367,8
9,community_to_narrative,2367,4


,unique_entity_labels,unique_event_labels,unique_docs_in_entities,communities,narratives
0,14920,24234,2426,2367,62


## Extraction Preview

In [5]:
preview(tables["entities"], ["doc_id", "paragraph_id", "label", "type", "description", "confidence"], n=15)
preview(tables["events"], ["doc_id", "paragraph_id", "label", "type", "date", "location", "confidence"], n=15)
preview(tables["claims"], ["doc_id", "paragraph_id", "claim_text", "speaker", "target", "stance", "confidence"], n=15)
preview(tables["relations"], ["doc_id", "paragraph_id", "source", "target", "relation", "confidence"], n=15)


,doc_id,paragraph_id,label,type,description,confidence
0,3175206_CIA-RDP79T00975A000100190001-9,p0,United States,Country,Country involved in lead supply negotiations.,0.7
1,3175206_CIA-RDP79T00975A000100190001-9,p0,United Kingdom,Country,Country seeking lead from the US.,0.7
2,3175206_CIA-RDP79T00975A000100190001-9,p0,USSR,Country,Country receiving lead shipments from London brokers.,0.7
3,3175206_CIA-RDP79T00975A000100190001-9,p0,Social Democratic Party,PoliticalGroup,Political party opposing the Schuman Plan.,0.7
4,3175206_CIA-RDP79T00975A000100190001-9,p0,Chancellor Adenauer,Person,Chancellor of West Germany involved in the Schuman Plan.,0.7
5,3175206_CIA-RDP79T00975A000100190001-9,p0,Czech Communist Party,PoliticalGroup,Party where Titoist movement was uncovered.,0.7
6,3175206_CIA-RDP79T00975A000100190001-9,p0,Wonsan,Location,Location of North Korean mine-laying operations.,0.7
7,3175206_CIA-RDP79T00975A000100190001-9,p0,Sohojin,Location,Location near Hungnam where mine-laying crews are requested to continue work.,0.7
8,3175206_CIA-RDP79T00975A000100190001-9,p0,Bundestag,Government,German federal parliament where the Schuman Plan is to be submitted.,0.7
9,3175206_CIA-RDP79T00975A000100190001-9,p1,Vladimir Clementis,Person,"Former Foreign Minister of Czechoslovakia, arrested for organizing a plot within the Czech Communist Party.",0.7


,doc_id,paragraph_id,label,type,date,location,confidence
0,3175206_CIA-RDP79T00975A000100190001-9,p0,Mine-laying operations completed,MilitaryAction,1951-02-26,Wonsan,0.7
1,3175206_CIA-RDP79T00975A000100190001-9,p0,Opposition to Schuman Plan,PolicyAction,NaN,NaN,0.7
2,3175206_CIA-RDP79T00975A000100190001-9,p1,Arrest of Vladimir Clementis,Arrest,NaN,Czech Republic,0.7
3,3175206_CIA-RDP79T00975A000100190001-9,p1,Address to the Central Committee,Speech,NaN,Czech Republic,0.7
4,2682857_CIA-RDP79T00975A000100210001-6,p0,US-Iceland defense negotiations,Negotiation,1951-03-02,Iceland,0.7
5,2682857_CIA-RDP79T00975A000100210001-6,p0,Pleven's resignation,Other,1951-03-02,France,0.7
6,2682857_CIA-RDP79T00975A000100210001-6,p1,Demonstration against US Assistant Secretary Miller,Demonstration,1951-03-05,Santiago,0.7
7,2682857_CIA-RDP79T00975A000100210001-6,p1,Inter-American Meeting of Foreign Ministers,Meeting,NaN,NaN,0.7
8,2682857_CIA-RDP79T00975A000100210001-6,p1,Economic cooperation discussion,EconomicAction,NaN,NaN,0.7
9,2682857_CIA-RDP79T00975A000100210001-6,p2,Brazil's intention to send troops to Korea,MilitaryAction,NaN,Korea,0.7


,doc_id,paragraph_id,claim_text,speaker,target,stance,confidence
0,3175206_CIA-RDP79T00975A000100190001-9,p0,"The British Ministry of Supply is seeking between 2,000 and 3,000 tons of lead for prompt shipment from the US.",British Ministry of Supply,United States,support,0.7
1,3175206_CIA-RDP79T00975A000100190001-9,p0,The Social Democratic Party will oppose the Schuman Plan categorically.,Social Democratic Party member,Schuman Plan,oppose,0.7
2,3175206_CIA-RDP79T00975A000100190001-9,p1,Czechoslovakia will not become a second Yugoslavia.,Gottwald,Czechoslovakia,support,0.7
3,3175206_CIA-RDP79T00975A000100190001-9,p1,There is a struggle for influence within the Czech Party.,Commentator,Czech Communist Party,neutral,0.7
4,2682857_CIA-RDP79T00975A000100210001-6,p0,Iceland's acceptance of foreign troops in peacetime would constitute the greatest departure from its traditional isolationism.,Commentator,Iceland,neutral,0.7
5,2682857_CIA-RDP79T00975A000100210001-6,p0,Pleven's resignation is an anti-Communist maneuver.,US Embassy in Paris,Pleven,support,0.7
6,2682857_CIA-RDP79T00975A000100210001-6,p1,The Socialist Party will take the 'popular line' that unification and neutralization of East and West Germany are theoretically possible.,Schumacher,Socialist Party,support,0.7
7,2682857_CIA-RDP79T00975A000100210001-6,p1,The Socialist Party does not believe in the practicability of a neutral Germany.,Commentary,Socialist Party,oppose,0.7
8,2682857_CIA-RDP79T00975A000100210001-6,p1,Brazil's cooperation will be on a quid pro quo basis.,Commentary,Brazil,neutral,0.7
9,2682857_CIA-RDP79T00975A000100210001-6,p2,Brazil could not meet the requirements of the Unified Command for troops sent to Korea,Brazilian military officials and former Foreign Minister Fernandes,Unified Command,oppose,0.7


,doc_id,paragraph_id,source,target,relation,confidence
0,3175206_CIA-RDP79T00975A000100190001-9,p0,United Kingdom,United States,REQUEST,0.7
1,3175206_CIA-RDP79T00975A000100190001-9,p0,Social Democratic Party,Schuman Plan,OPPOSITION,0.7
2,3175206_CIA-RDP79T00975A000100190001-9,p0,Chancellor Adenauer,Bundestag,REQUEST,0.7
3,3175206_CIA-RDP79T00975A000100190001-9,p1,Vladimir Clementis,Czech Communist Party,AFFILIATION,0.7
4,3175206_CIA-RDP79T00975A000100190001-9,p1,Gottwald,Czech Communist Party,EVENT_PARTICIPANT,0.7
5,2682857_CIA-RDP79T00975A000100210001-6,p0,Benediktsson,Icelandic Government,AFFILIATION,0.7
6,2682857_CIA-RDP79T00975A000100210001-6,p0,Pleven,Auriol,COMMUNICATION,0.7
7,2682857_CIA-RDP79T00975A000100210001-6,p0,US Embassy in Paris,Pleven,MENTIONS,0.7
8,2682857_CIA-RDP79T00975A000100210001-6,p1,Schumacher,Socialist Party,AFFILIATION,0.7
9,2682857_CIA-RDP79T00975A000100210001-6,p1,Neves da Fontoura,US,SUPPORT,0.7


## Graph Preview

In [6]:
preview(tables["canonical_nodes"], ["node_key", "label", "node_type", "mention_count", "source_type"], n=20)
preview(tables["analysis_edges"], ["source", "target", "rel", "weight", "support_count"], n=20)
preview(tables["level1_summary"], ["level1_id", "display_label", "size", "top_labels", "top_types"], n=20)
preview(tables["level2_summary"], ["level2_id", "display_label", "size", "top_labels", "top_types"], n=20)


,node_key,label,node_type,mention_count,source_type
0,CANONICAL_ENTITY:10TH_AIR_ARMY__06D70836F3,10TH AIR ARMY [CZECHOSLOVAKIA / CLAIRE],CanonicalEntity,2.0,Military
1,CANONICAL_ENTITY:10TH_AIR_ARMY__526483EF40,10TH AIR ARMY [PARIS / MAJLIS],CanonicalEntity,1.0,Military
2,CANONICAL_ENTITY:10TH_AIR_ARMY__81860189AE,10TH AIR ARMY [PRAVDA / MONGOLIA],CanonicalEntity,1.0,Military
3,CANONICAL_ENTITY:10TH_AIR_DIVISION__403960816E,10TH AIR DIVISION,CanonicalEntity,1.0,Military
4,CANONICAL_ENTITY:10TH_ARTILLERY_DIVISION__0EFA616A8E,10TH ARTILLERY DIVISION,CanonicalEntity,1.0,Military
5,CANONICAL_ENTITY:11TH_ARMORED_INFANTRY_BRIGADE__94559B74DF,11TH ARMORED INFANTRY BRIGADE,CanonicalEntity,2.0,Military
6,CANONICAL_ENTITY:11TH_ARMY__42775C5DEE,11TH ARMY,CanonicalEntity,1.0,Military
7,CANONICAL_ENTITY:11TH_INTER_AMERICAN_CONFERENCE__665508FAF2,11TH INTER AMERICAN CONFERENCE,CanonicalEntity,1.0,Document
8,CANONICAL_ENTITY:11_MEMBER_EXECUTIVE_COUNCIL__EBEAC36E1C,11 MEMBER EXECUTIVE COUNCIL,CanonicalEntity,1.0,Organization
9,CANONICAL_ENTITY:12TH_ARMY__31678428BF,12TH ARMY,CanonicalEntity,1.0,Military


,rel,weight,support_count
0,CO_OCCURRENCE,0.075,1
1,CO_OCCURRENCE,0.075,1
2,CO_OCCURRENCE,0.075,1
3,CO_OCCURRENCE,0.075,1
4,CO_OCCURRENCE,0.075,1
5,CO_OCCURRENCE,0.075,1
6,CO_OCCURRENCE,0.075,1
7,CO_OCCURRENCE,0.075,1
8,CO_OCCURRENCE,0.075,1
9,CO_OCCURRENCE,0.075,1


,level1_id,display_label,size,top_labels,top_types
0,0,Geopolitical Dynamics in the Middle East,2343,EGYPT | ISRAEL | JORDAN | SYRIA | BEN GURION | KING HUSSAIN | LEBANON | UAR [AB96],"{""EventCluster"": 1392, ""CanonicalEntity"": 951}"
1,1,Laos and Regional Political Dynamics,1771,SOUVANNA PHOUMA | LAOS | GENERAL PHOUMI | PATHET LAO [AB93] | VIENTIANE | KONG LE | NORTH VIETNAM | SOUPHANNOUVONG,"{""EventCluster"": 1084, ""CanonicalEntity"": 687}"
2,2,Southeast Asia Military Dynamics,1619,VIET MINH [A5CF] | VIET MINH [DIEN / PHUNAVARRE] | DIEM | BAO DAI | CAMBODIA | SOUTH VIETNAM | SIHANOUK | Viet Minh attack on Laos,"{""EventCluster"": 885, ""CanonicalEntity"": 734}"
3,3,Geopolitical Dynamics in Central Africa,1559,CONGO | GIZENGA | LUMUMBA | MOBUTU | BELGIUM | KASAVUBU | TSHOMBÉ | HAMMARSKJOLD,"{""EventCluster"": 908, ""CanonicalEntity"": 651}"
4,4,Geopolitical Dynamics of Iranian Oil,1537,IRAN | MOSSADEQ | SHAH OF IRAN | SHAH [71AD] | ZAHEDI | AMBASSADOR HENDERSON | ANGLO IRANIAN OIL COMPANY | TEHRAN,"{""EventCluster"": 832, ""CanonicalEntity"": 705}"
5,5,Indonesian Political Leadership and Dynamics,1510,SUKARNO [INDONESIA] | INDONESIA | GENERAL NASUTION | SUKARNO [BD7E] | HATTA | INDONESIAN GOVERNMENT [A63D] | MASJUMI [A2A8] | NATIONAL P...,"{""EventCluster"": 820, ""CanonicalEntity"": 690}"
6,6,Geopolitical Dynamics of North Africa and France,1469,FRANCE | DE GAULLE | ALGERIA | TUNISIA | BOURGUIBA | ALGIERS | MOLLET | ALGERIAN REBELS [IVORY / BOIGNY],"{""EventCluster"": 750, ""CanonicalEntity"": 719}"
7,7,Geopolitical Dynamics in Southeastern Europe,1467,YUGOSLAVIA | ITALY | GREECE | TITO | TRIESTE | DE GASPERI | BULGARIA | PELLA,"{""EventCluster"": 791, ""CanonicalEntity"": 676}"
8,8,Post-War Political Dynamics in Europe,1384,ADENAUER | CHANCELLOR ADENAUER | CHRISTIAN DEMOCRATIC UNION | CHRISTIAN DEMOCRATIC PARTY | SOCIAL DEMOCRATIC PARTY | CHRISTIAN DEMOCRATS...,"{""CanonicalEntity"": 729, ""EventCluster"": 655}"
9,9,Cold War Dynamics in South Asia,1314,INDIA | NEHRU | PEIPING [8594] | CHOU EN LAI | Chou's visit to Rangoon | DALAI LAMA | NEPAL | NEW DELHI,"{""EventCluster"": 742, ""CanonicalEntity"": 572}"


,level2_id,display_label,size,top_labels,top_types
0,0,Cease-fire agreement / Exchange of fire / Demonstrations in Algeria / Bourguiba's speech,19,Cease-fire agreement / Exchange of fire | Demonstrations in Algeria / Bourguiba's speech | Troop Withdrawal / Khrushchev's speech | Nego...,"{""LEVEL1_COMMUNITY"": 19}"
1,1,Negotiations in Laos / Press Conference / Visit to Moscow / Nehru's Statements,17,Negotiations in Laos / Press Conference | Visit to Moscow / Nehru's Statements | Recognition of Mongolia / Military assessment | Tananar...,"{""LEVEL1_COMMUNITY"": 17}"
2,2,Demonstrations in Trieste / Tito's speech / Khrushchev's speech / Geneva Conference,14,Demonstrations in Trieste / Tito's speech | Khrushchev's speech / Geneva Conference | Cabinet Meeting / Bundestag debate | Cabinet Meeti...,"{""LEVEL1_COMMUNITY"": 14}"
3,3,Rioting in Caracas / Foray into Nicaragua / Demonstrations in Cuba / Arrest of Huber Matos,9,Rioting in Caracas / Foray into Nicaragua | Demonstrations in Cuba / Arrest of Huber Matos | 62 GROUP / 8TH AIR DIVISION | Anti-US demon...,"{""LEVEL1_COMMUNITY"": 9}"
4,4,BERLIN LEADERS / FEDERAL GOVERNMENT,1,BERLIN LEADERS / FEDERAL GOVERNMENT,"{""LEVEL1_COMMUNITY"": 1}"
5,5,AMERICAN OFFICIALS [COMMANDANTS / BERLIN] / BRITISH BERLIN COMMANDANTS,1,AMERICAN OFFICIALS [COMMANDANTS / BERLIN] / BRITISH BERLIN COMMANDANTS,"{""LEVEL1_COMMUNITY"": 1}"
6,6,ARMY [LED / PEASANT] / COMMUNIST LED LABOR AND PEASANT GROUPS,1,ARMY [LED / PEASANT] / COMMUNIST LED LABOR AND PEASANT GROUPS,"{""LEVEL1_COMMUNITY"": 1}"


## Hierarchy Preview

In [7]:
preview(tables["community_nodes"], ["community_id", "label", "analyst_label", "size", "top_labels", "analyst_summary"], n=20)
preview(tables["narrative_nodes"], ["narrative_id", "label", "analyst_label", "num_communities", "top_labels", "analyst_summary"], n=20)
preview(tables["hierarchy_index"], ["community_id", "community_label", "narrative_id", "narrative_label", "community_size"], n=30)


,community_id,label,analyst_label,size,top_labels,analyst_summary
0,1,Jordanian Political and Military Leadership,Middle Eastern Governance and Diplomacy,148,KING HUSSAIN | NUWAR [LEGION] | NABULSI | AMBASSADOR MALLORY | SAMIR RIFAI | PRIME MINISTER NABULSI | AMMAN | HIYARI,This community encompasses key figures and events in Jordan's political and military landscape during a pivotal historical period.
1,135,Sihanouk's Political Influence in Cambodia,Cambodian Political Dynamics and Leadership,139,SIHANOUK | CAMBODIA | DAP CHHUON | SON NGOC THANH | SAM SARY | MONGOLIA | KING SURAMARIT | Resignation of Prince Sihanouk,This community explores the political impact of Sihanouk and key figures in Cambodia's history.
2,2,Ben Gurion and Political Allies,Israeli Political Leadership and Events,102,BEN GURION | LAWSON | SHARETT | Ben-Gurion's visit to the US | MAPAI [CD91] | MAPAI PARTY [1989] | PINHAS LAVON | ACHDUT HAAVODA,This community focuses on Ben Gurion and his political associates during key events in Israeli history.
3,621,Soviet Influence in International Affairs,Cold War Geopolitical Dynamics,89,SOVIET UNION [FFC2] | ADMIRAL MARTADINATA | GENEVA AIR SHOW | AIR MARSHAL SURYADARMA | Soviet Fighter Attack | NORWEGIAN OFFICIALS | Sov...,This community explores the Soviet Union's impact on global events and international relations during the Cold War.
4,3,Lebanon's Energy and Economic Dynamics,Middle Eastern Energy Politics and Infrastructure,86,LEBANON | TRANS ARABIAN PIPELINE COMPANY | Conference on revenue demands | KARAMI | Attack on Wingfoot estate | Lebanon's Acceptance of ...,This community explores Lebanon's role in energy negotiations and economic strategies related to the Trans Arabian Pipeline.
5,285,Dutch Political Dynamics and Maritime Affairs,European Governance and Naval Operations,82,NETHERLANDS | DUTCH | Cabinet crisis in Finland | POLISH MERCHANT MARINE | Troop commitment for Korea | Coal trains stopped | Arrival of...,This community focuses on the interplay of Dutch political events and maritime activities.
6,145,South Vietnamese Military Leadership Dynamics,Military and Political Power Struggles in Vietnam,80,DIEM | GENERAL HINH | CAO DAI [DC94] | HOA HAO SECT | GENERAL NGUYEN KHANH | GENERAL DUONG VAN MINH | Attempted coup against Diem | SOUT...,This community focuses on the interactions and conflicts among South Vietnamese military leaders during the Diem regime.
7,519,Geopolitical Tensions in Peiping,Historical Conflicts and Diplomatic Relations,80,PEIPING [8594] | Issuance of warning | US WARSHIPS | Visit to Peiping | ALL UNION ASSOCIATION FOR PETROLEUM PRODUCTS | GERMAN LUFTWAFFE ...,This community focuses on the geopolitical dynamics and events surrounding Peiping during periods of conflict and diplomatic activity.
8,639,Political Dynamics in South Asian Leadership,Historical Political Movements in South Asia,80,MIRZA | SUHRAWARDY | AWAMI LEAGUE | REPUBLICAN PARTY [YOSHIO / SHIGA] | I I CHUNDRIGAR | CHUNDRIGAR | PRIME MINISTER NOON | MAULANA BHAS...,This community explores the interplay of key political figures and parties in South Asia's historical landscape.
9,234,Diplomatic Engagements in Oil Negotiations,Energy Diplomacy and Geopolitical Strategies,79,AMBASSADOR HENDERSON | Oil agreement | SENATE [MAJLIS / HENDERSON] | Oil negotiations begin | IRANIAN CABINET | Defense policy readjustm...,This community focuses on the interplay of diplomacy and oil agreements in shaping regional defense policies.


,narrative_id,label,analyst_label,num_communities,top_labels,analyst_summary
0,0,Geopolitical Dynamics in the Middle East,Middle Eastern Political and Military Interactions,73,EGYPT | ISRAEL | JORDAN | SYRIA | BEN GURION | KING HUSSAIN | LEBANON | UAR [AB96],This narrative explores the complex political and military relationships among key Middle Eastern nations.
1,7,Geopolitical Dynamics in Southeastern Europe,Regional Political Interactions and Alliances,64,YUGOSLAVIA | ITALY | GREECE | TITO | TRIESTE | DE GASPERI | BULGARIA | PELLA,Exploring the complex diplomatic relationships and political developments in Southeastern Europe during the 20th century.
2,28,Geopolitical Alliances and Tensions,International Political Dynamics,64,UNITED STATES [41F6] | BRITAIN | ICELAND | APRA PARTY | REYKJAVIK | LABOR ALLIANCE | ODRIA | NATO [ICELAND / BRITAIN],This narrative explores the complex relationships and conflicts among nations in the context of political alliances and diplomatic negot...
3,6,Geopolitical Dynamics of North Africa and France,Cold War Era International Relations,63,FRANCE | DE GAULLE | ALGERIA | TUNISIA | BOURGUIBA | ALGIERS | MOLLET | ALGERIAN REBELS [IVORY / BOIGNY],This narrative explores the complex interactions between France and North African nations during the Cold War.
4,21,Geopolitical Tensions in Divided Germany,Cold War Divisions and Alliances,62,EAST GERMANY | WEST GERMANY | WEST BERLIN | BERLIN | EAST BERLIN | SOCIALIST UNITY PARTY [F7EB] | WALTER ULBRICHT | BONN GOVERNMENT [864D],This narrative explores the complex geopolitical dynamics and tensions in Germany during the Cold War era.
5,11,Cold War Geopolitical Dynamics in South Asia,Geopolitical Influences and Leadership Changes,59,PAKISTAN | AFGHANISTAN | SOVIET UNION [FFC2] | MIRZA | AYUB | DAUD | NAIM | SUHRAWARDY,This narrative explores the complex interplay of Cold War politics and leadership shifts in South Asia.
6,9,Cold War Dynamics in South Asia,Geopolitical Interactions in South Asia,58,INDIA | NEHRU | PEIPING [8594] | CHOU EN LAI | Chou's visit to Rangoon | DALAI LAMA | NEPAL | NEW DELHI,This narrative explores the complex geopolitical relationships and conflicts in South Asia during the Cold War era.
7,4,Geopolitical Dynamics of Iranian Oil,Energy Politics and Regional Alliances,57,IRAN | MOSSADEQ | SHAH OF IRAN | SHAH [71AD] | ZAHEDI | AMBASSADOR HENDERSON | ANGLO IRANIAN OIL COMPANY | TEHRAN,This narrative explores the complex interplay of Iranian oil politics and its impact on regional and global diplomatic relations.
8,2,Southeast Asia Military Dynamics,Southeast Asian Political and Military Interactions,54,VIET MINH [A5CF] | VIET MINH [DIEN / PHUNAVARRE] | DIEM | BAO DAI | CAMBODIA | SOUTH VIETNAM | SIHANOUK | Viet Minh attack on Laos,This narrative explores the complex political and military relationships in Southeast Asia during the Cold War era.
9,10,Cold War Diplomacy and Governance,Geopolitical Strategies and Leadership Dynamics,52,KHRUSHCHEV | USSR [4A23] | BULGANIN | EISENHOWER | MOLOTOV | MALENKOV | MIKOYAN | AMBASSADOR THOMPSON,Explores the intricate diplomatic engagements and governance strategies during the Cold War era.


,community_id,community_label,narrative_id,narrative_label,community_size
0,0,Beirut and Mixed Armistice Dynamics,0,Geopolitical Dynamics in the Middle East,44
1,1,Jordanian Political and Military Leadership,0,Geopolitical Dynamics in the Middle East,148
2,2,Ben Gurion and Political Allies,0,Geopolitical Dynamics in the Middle East,102
3,3,Lebanon's Energy and Economic Dynamics,0,Geopolitical Dynamics in the Middle East,86
4,4,Israeli Military Dynamics Overview,0,Geopolitical Dynamics in the Middle East,55
5,5,Jordan Border Incident Analysis,0,Geopolitical Dynamics in the Middle East,12
6,6,Alexandria and Regional Military Engagements,0,Geopolitical Dynamics in the Middle East,55
7,7,Syrian Military and Political Dynamics,0,Geopolitical Dynamics in the Middle East,74
8,8,Israeli Government Leadership and Actions,0,Geopolitical Dynamics in the Middle East,35
9,9,Aswan Dam Negotiations and Political Maneuvering,0,Geopolitical Dynamics in the Middle East,19


## Visualization Files

In [8]:
vis_paths = [
    PROJECT_DIR / "hierarchy" / "visualizations" / "community_graph_pyvis.html",
    PROJECT_DIR / "hierarchy" / "visualizations" / "narrative_graph_pyvis.html",
]

for path in vis_paths:
    exists = path.exists()
    display(Markdown(f"- `{rel(path, REPO_ROOT)}`: {'present' if exists else 'missing'}"))

if SHOW_HTML_VISUALIZATIONS:
    for path in vis_paths:
        if path.exists():
            display(Markdown(f"### {path.name}"))
            display(HTML(path.read_text(encoding="utf-8")))


- `outputs/kg_cia_full_para_v2/hierarchy/visualizations/community_graph_pyvis.html`: present

- `outputs/kg_cia_full_para_v2/hierarchy/visualizations/narrative_graph_pyvis.html`: present